# Brain Tumor MRI: train on Google Colab

Clones **[Tumor-AI-training-model](https://github.com/MuhammadSaljooq/Tumor-AI-training-model)** (`checkpoint_added` by default), installs deps, prepares data, runs training.

1. **Runtime > Change runtime type > GPU** (recommended).
2. **Data:** Kaggle (upload `kaggle.json`) **or** Google Drive folder with `Training/` and `Testing/`.

Edit **User settings** in the next cell. First run: use `resnet50` and low epochs; hybrid needs more VRAM/time.


In [ ]:
# --- User settings ---
REPO_URL = "https://github.com/MuhammadSaljooq/Tumor-AI-training-model.git"
BRANCH = "checkpoint_added"
PROJECT_DIR = "/content/Tumor-AI-training-model"

CONFIG_FILE = "configs/config_colab.yaml"
MODELS = "resnet50"
EXTRA_ARGS = ""

DATA_SOURCE = "kaggle"
DRIVE_DATA_PATH = "/content/drive/MyDrive/brain_tumor_data"


In [ ]:
!nvidia-smi


In [ ]:
import os
import shutil
import subprocess
import sys
from pathlib import Path


def run(cmd, cwd=None):
    print("+", cmd if isinstance(cmd, str) else " ".join(cmd))
    subprocess.check_call(cmd, cwd=cwd)


if Path(PROJECT_DIR).exists():
    print("Updating repo...")
    run(["git", "-C", PROJECT_DIR, "fetch", "origin", BRANCH])
    run(["git", "-C", PROJECT_DIR, "checkout", BRANCH])
    run(["git", "-C", PROJECT_DIR, "pull", "origin", BRANCH])
else:
    run(["git", "clone", "--branch", BRANCH, "--depth", "1", REPO_URL, PROJECT_DIR])

os.chdir(PROJECT_DIR)
run([sys.executable, "-m", "pip", "install", "-q", "-U", "pip"])
run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"])
run([sys.executable, "-m", "pip", "install", "-q", "kaggle"])
print("Project:", PROJECT_DIR)


In [ ]:
import os
import shutil
import subprocess
import zipfile
from pathlib import Path

raw = Path(PROJECT_DIR) / "data" / "raw"
raw.mkdir(parents=True, exist_ok=True)

if DATA_SOURCE == "kaggle":
    from google.colab import files
    print("Upload kaggle.json (Kaggle account -> API -> Create New Token)")
    uploaded = files.upload()
    kdir = Path.home() / ".kaggle"
    kdir.mkdir(exist_ok=True)
    found = False
    for name, data in uploaded.items():
        if name.endswith(".json"):
            (kdir / "kaggle.json").write_bytes(data)
            found = True
            break
    if not found:
        raise RuntimeError("Upload kaggle.json")
    os.chmod(kdir / "kaggle.json", 0o600)

    data_parent = Path(PROJECT_DIR) / "data"
    zip_path = data_parent / "brain-tumor-mri-dataset.zip"
    subprocess.run(
        [
            "kaggle", "datasets", "download",
            "-d", "masoudnickparvar/brain-tumor-mri-dataset",
            "-p", str(data_parent),
        ],
        check=True,
        cwd=PROJECT_DIR,
    )
    with zipfile.ZipFile(zip_path, "r") as z:
        z.extractall(raw)
    zip_path.unlink(missing_ok=True)

elif DATA_SOURCE == "drive":
    from google.colab import drive
    drive.mount("/content/drive")
    src = Path(DRIVE_DATA_PATH)
    if not src.is_dir():
        raise FileNotFoundError(DRIVE_DATA_PATH)
    for item in src.iterdir():
        dest = raw / item.name
        if dest.exists():
            shutil.rmtree(dest) if dest.is_dir() else dest.unlink()
        if item.is_dir():
            shutil.copytree(item, dest)
        else:
            shutil.copy2(item, dest)
else:
    print("DATA_SOURCE already_cloned: put dataset under", raw)

subs = [p for p in raw.iterdir() if p.is_dir() and p.name not in {".ipynb_checkpoints"}]
if len(subs) == 1 and (subs[0] / "Training").is_dir():
    inner = subs[0]
    for item in inner.iterdir():
        shutil.move(str(item), str(raw / item.name))
    inner.rmdir()
    print("Flattened:", raw)

if (raw / "Training").is_dir():
    print("OK: Training/ under data/raw")
else:
    print("WARNING: expected data/raw/Training for Kaggle layout")


In [ ]:
import os
import shlex
import subprocess
import sys

os.chdir(PROJECT_DIR)
cmd = [sys.executable, "main.py", "--config", CONFIG_FILE, "--models", *MODELS.split()]
if EXTRA_ARGS.strip():
    cmd += shlex.split(EXTRA_ARGS.strip())
print("Running:", " ".join(cmd))
subprocess.run(cmd, check=True, cwd=PROJECT_DIR)


## Outputs

- `results/checkpoints/` (`*_best.pth`, `*_last.pth`)
- `results/plots/`, `results/final_results.csv`

Zip and download:

```python
from google.colab import files
!cd /content/Tumor-AI-training-model && zip -r /content/artifacts.zip results/checkpoints results/plots results/final_results.csv 2>/dev/null || true
files.download("/content/artifacts.zip")
```
